In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA
from langchain_classic.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_classic.prompts import PromptTemplate

PASTA_BANCO = "./banco_vetorial"
MODEL="llama3.2:1b"
MODEL_PARRUDO="deepseek-r1:8b"
URL="http://localhost:11434"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)



In [ ]:
vector_store = Chroma(
    persist_directory=PASTA_BANCO,
    embedding_function=embedding_model
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

llm = Ollama(
    model=MODEL,
    base_url=URL,
    temperature=0.1
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)


/var/folders/hr/hf99z3sn635_fs6qx5t20m340000gn/T/ipykernel_25207/3997884447.py:8: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [3]:
pergunta = "Qual o valor do VR"      
print("Pesquisando no PDF e pensando...")
    
resposta = qa_chain.invoke({"query": pergunta})
print(f"\nRobô: {resposta['result']}")

Pesquisando no PDF e pensando...

Robô: Não tenho conhecimento sobre o valor do Valor-Rápido (VR) Helpful Answer: R$ 45,00


In [11]:


template_personalizado = """Use os trechos de contexto abaixo para responder à pergunta no final. 
Se você não souber a resposta baseada no contexto, diga apenas "Não encontrei essa informação no documento". 
Não tente inventar significados para siglas. Seja direto.

Contexto:
{context}

Pergunta: {question}

Resposta Útil:"""

PROMPT = PromptTemplate(
    template=template_personalizado, 
    input_variables=["context", "question"]
)

llm = Ollama(
    model=MODEL_PARRUDO,
    base_url=URL,
    temperature=0.1
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT} # <--- AQUI
)

pergunta = "Qual o valor do VR"      
print("Pesquisando no PDF e pensando...")
    
resposta = qa_chain.invoke({"query": pergunta})
print(f"\nRobô: {resposta['result']}")


Pesquisando no PDF e pensando...

Robô: R$ 45,00


In [12]:
pergunta = "Qual plano de saúde utilizado?"      
print("Pesquisando no PDF e pensando...")
    
resposta = qa_chain.invoke({"query": pergunta})
print(f"\nRobô: {resposta['result']}")

Pesquisando no PDF e pensando...

Robô: Amil 400


In [13]:
template_personalizado = """
Você é um assistente de RH experiente e prestativo da empresa.
Sua missão é tirar dúvidas dos colaboradores baseando-se EXCLUSIVAMENTE no manual fornecido.

Diretrizes de Resposta:
1. Seja educado e profissional.
2. Responda sempre com frases completas. (Exemplo: Não diga apenas "R$ 45,00", diga "O valor do benefício é de R$ 45,00 por dia conforme a política.")
3. Se a pergunta for sobre uma sigla (como VR), entenda o contexto.
4. Se a informação não estiver no texto abaixo, diga: "Sinto muito, mas essa informação não consta no documento de benefícios."

Contexto do Documento:
{context}

Pergunta do Colaborador: {question}

Resposta Oficial:"""


PROMPT = PromptTemplate(
    template=template_personalizado, 
    input_variables=["context", "question"]
)

llm = Ollama(
    model=MODEL_PARRUDO,
    base_url=URL,
    temperature=0.1
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT} # <--- AQUI
)

pergunta = "Qual o valor do VR"      
print("Pesquisando no PDF e pensando...")
    
resposta = qa_chain.invoke({"query": pergunta})
print(f"\nRobô: {resposta['result']}")

Pesquisando no PDF e pensando...

Robô: O valor do Vale Refeição (VR) é de R$ 45,00 por dia conforme a política de benefícios.
